The contents of this directory benchmark various model approaches. Results are saved as
1. Actual models
2. Slurm logs
3. HTML Dask performance reports

# Imports

In [5]:
# Imports
import subprocess
from datetime import datetime
import pickle
import os

import numpy as np
import pandas as pd

import statsmodels
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialResultsWrapper as smzinb

# Submitting model runs

In [6]:
#functions for model submission...

data_root="/gpfs/gibbs/pi/reilly/tabula_data"

def bench(model_code, t):
    """
    Executes a particular model design & collects statistics. 
    t is time in hours
    """
    now=datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    logdir=f"{data_root}/speed_test/logs/{model_code}_{now}"

    #make a directory to put all the log files : this will be 
    os.makedirs(logdir, exist_ok=True)

    command=f"""
    module load miniconda
    conda activate env_tensorzinb_cuda
    code_location=$(pwd)
    cd {logdir}
    python ${{code_location}}/cluster.py {model_code}
    """


    slurm_cmd = [
        "sbatch",
        "--partition=ycga",
        f"--time={t}:00:00",
        f"--output={logdir}/master_{model_code}_{now}.out",
        "-c 1",
        "--mem-per-cpu=15G",
        f"-J {model_code}_master",
        "--wrap", command
    ]
    result = subprocess.run(
        slurm_cmd, 
        capture_output=True, 
        text=True
    )
    
    print(result.stdout.strip() if result.returncode == 0 else result.stderr.strip())



In [7]:
#bench("c0011020",4)

# Summarizing success / failure
& everything in-between

In [8]:
def extract_parameters():
    """
    Takes a model & extracts triplets (pi, sigma-squared, mu)
    (break by type...)
    unimplemented
    """
    pass

def load_model(model_file:str):
    """
    What is says on the tin. Points to relevant archive path to avoid retyping.
    """
    print(f"[+] loading {model_file}")
    with open(f"{data_root}/speed_test/models/arch/{model_file}","rb") as f:
        return pickle.load(f)

def eval_single_statsmodels(model):
    """
    Takes a single statsmodels model & returns some useful QC metrics
    """
    if model=="too_small":
        return {
            "converged": 0,
            "too_small":1,
            "covariance_matrix_available": None,
            "covariance_matrix_finite": None,
            "standard_errors_finite": None
        }

    cov=-1
    finite_cov=-1
    try:
        cov = model.cov_params()
        finite_cov = np.all(np.isfinite(cov.values))
    except (ValueError, np.linalg.LinAlgError) as e:
        finite_cov = False
        cov = None

    converged = model.mle_retvals.get("converged", False)
    finite_se = np.all(np.isfinite(model.bse)) if hasattr(model, "bse") else False

    return {
        "converged": converged,
        "too_small":0,
        "covariance_matrix_available": cov is not None,
        "covariance_matrix_finite": finite_cov,
        "standard_errors_finite": finite_se
    }

def infer_converged_from_weights(model, tolerance=1e-3):
    all_weights = []
    for w in model.get('weights', {}).values():
        if isinstance(w, np.ndarray):
            all_weights.append(w.flatten())
    
    if not all_weights:
        return False  # no weights found, assume not converged

    all_weights = np.concatenate(all_weights)

    std_dev = np.std(all_weights)

    return int(std_dev > tolerance)

def eval_single_tensorzinb(model):
    """
    Takes a single tensorzinb model & returns some useful QC metrics. 
    """
    if model=="too_small":
        return {
            "converged": 0,
            "too_small":1,
            "covariance_matrix_available": None,
            "covariance_matrix_finite": None,
            "standard_errors_finite": None
        }

    return {
        "converged": infer_converged_from_weights(model),
        "too_small":0,
        "covariance_matrix_available": None,
        "covariance_matrix_finite": None,
        "standard_errors_finite": None
    }

def eval_model(model):
    """
    Produces QC metrics on a single model, regardless of type
    """
    #type checking is done here
    
    if isinstance(model,smzinb):
        #if its a statsmodels object, it must be a unified statsmodels model
        return pd.DataFrame.from_dict(eval_single_statsmodels(model), orient='index', columns=['only'])
        
    elif isinstance(model,dict):
        #if it's a dict... it could be a broken statsmodels
        # or unified or broken 
        #
        all_zinb = [
            isinstance(model[key], smzinb) or isinstance(model[key], str)
            for key in model
        ]
        if all(all_zinb):
            #must be broken statsmodels
            return pd.DataFrame({
                key:eval_single_statsmodels(model[key])
                for key in model
            })

        else:
            #must be either unified or broken tensorzinb.
            #disambiguate by checking for a sentinel value
            if "multi_sent" in model.keys():
                del model["multi_sent"]
                # broken tensorzinb
                return pd.DataFrame({
                    key:eval_single_tensorzinb(model[key])
                    for key in model
                })
                print("[!] Type not implemented yet. Aborting.")
            else:
                #single tensorzinb model. 
                return pd.DataFrame.from_dict(
                    eval_single_tensorzinb(model),
                    orient='index'
                )
                
    else:
        print("[!] Type not implemented yet. Aborting.")



def eval_models(modelpaths):
    """
    Loads and evaluates models, taking a list of filenames
    and returning a dictionary of model IDs pointing at
    evaluations
    """
    evals={}
    for path in modelpaths:
        name=path.split("_")[0]
        try:
            model=load_model(path)
            evals[name]=eval_model(model)
            del model
        except EOFError:
            print(f"EOF error on {name}")
    return evals

def compare_models(evals):
    """
    Takes output of eval_models and compresses
    into a summary dataframe. 
    """
    summaries=[]
    for name in evals:
        print(f"[+] Evaluating {name}")
        summary=evals[name].sum(axis=1)
        summary["total"]=len(evals[name].columns)
        summary["model"]=name
        summaries.append(summary)
    summaries=pd.DataFrame(summaries)
    summaries.set_index(summaries["model"],inplace=True)
    summaries.drop("model",axis=1,inplace=True)
    return summaries

def eval_all():
    file_names=[f for f in os.listdir(f"{data_root}/speed_test/models/arch/")]
    evals=eval_models(file_names)
    return evals

def summarize_all():
    return compare_models(eval_all())


In [9]:
summary=summarize_all()

[+] loading c0010010_2025-04-30_13-50-03.pkl
[+] loading c0010020_2025-04-30_16-03-12.pkl
[+] loading c0011000_2025-04-28_18-27-55.pkl
[+] loading c0011020_2025-04-30_18-00-30.pkl
[+] loading c0012010_2025-04-29_18-13-58_fixed.pkl
[+] loading c0100010_2025-04-25_17-44-37.pkl
[+] loading c0100020_2025-04-30_14-32-41.pkl
[+] loading c0200010_2025-04-30_16-52-39.pkl
[+] loading c020020_2025-04-30_14-17-28.pkl
[+] loading c9011010_2025-04-29_16-28-13_fixed.pkl
[+] loading c9011090_2025-04-28_19-02-07.pkl
[+] loading c9100010_2025-04-25_15-51-10.pkl
[+] loading c9100090_2025-04-25_17-40-15.pkl
[+] loading c9200010_2025-04-25_17-44-36.pkl
[+] loading c9200020_2025-04-30_13-56-19.pkl
[+] loading c9200090_2025-04-25_17-44-35.pkl
[+] Evaluating c0010010
[+] Evaluating c0010020
[+] Evaluating c0011000
[+] Evaluating c0011020
[+] Evaluating c0012010
[+] Evaluating c0100010
[+] Evaluating c0100020
[+] Evaluating c0200010
[+] Evaluating c020020
[+] Evaluating c9011010
[+] Evaluating c9011090
[+] Ev

In [10]:
summary

,converged,too_small,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total
model,,,,,,
c0010010,10.0,0.0,0.0,0.0,0.0,10.0
c0010020,209.0,3.0,0.0,0.0,0.0,212.0
c0011000,1.0,0.0,0.0,0.0,0.0,1.0
c0011020,209.0,3.0,0.0,0.0,0.0,212.0
c0012010,10.0,0.0,0.0,0.0,0.0,10.0
c0100010,10,0.0,0,0,False,10.0
c0100020,196,0.0,134,True,True,212.0
c0200010,10,0.0,6,True,True,10.0
c020020,195,0.0,177,True,True,212.0


Adding metadata so we don't have to memorize all the IDs.

In [11]:
modelspecs=pd.read_csv(f"{data_root}/speed_test/modelspecs.tsv",sep="\t")
modelspecs.set_index(modelspecs["code"],inplace=True)
modelspecs.drop("code",axis=1,inplace=True)

In [12]:
summary=summary.join(modelspecs,how="inner")
summary

,converged,too_small,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total,dataset,sm_optimizer,lib,hardware,dna,main_equ_type,z_equ_type,main_equ,z_equ,broken_by
c0010010,10.0,0.0,0.0,0.0,0.0,10.0,Shendure (0),0_default,tensorzinb (1),0_CPU,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c0010020,209.0,3.0,0.0,0.0,0.0,212.0,Shendure (0),0_default,tensorzinb (1),0_CPU,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id
c0011000,1.0,0.0,0.0,0.0,0.0,1.0,Shendure (0),0_default,tensorzinb (1),1_A100,No dna (0),simple addition (0),replicate (0),umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),NaN
c0011020,209.0,3.0,0.0,0.0,0.0,212.0,Shendure (0),0_default,tensorzinb (1),1_A100,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id
c0012010,10.0,0.0,0.0,0.0,0.0,10.0,Shendure (0),0_default,tensorzinb (1),2_A100,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c0100010,10,0.0,0,0,False,10.0,Shendure (0),1_bfgs,statsmodels (0),0_CPU,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c0100020,196,0.0,134,True,True,212.0,Shendure (0),1_bfgs,statsmodels (0),0_CPU,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id
c0200010,10,0.0,6,True,True,10.0,Shendure (0),2_cg,statsmodels (0),0_CPU,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c020020,195,0.0,177,True,True,212.0,Shendure (0),2_cg,statsmodels (0),0_CPU,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id
c9011010,2.0,0.0,0.0,0.0,0.0,2.0,Fake (9),0_default,tensorzinb (1),1_A100,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type


# Summarizing performance

In [13]:
def format_seconds(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f"{hours}h {minutes}m {secs:.2f}s"

def simple_time(model_file):
    """
    Produces a quick time estimate from log files.
    Not as accurate or detailed as the full HTML report.
    """
    directory=f"{data_root}/speed_test/logs/arch/{model_file}"
    master_files = [f for f in os.listdir(directory) if f.startswith("master")]
    if len(master_files) != 1:
        raise ValueError(f"Expected exactly one 'master*' file, found {len(master_files)}.")
    
    filepath = os.path.join(directory, master_files[0])

    # dump file into memory (its small)
    with open(filepath, 'r') as file:
        lines = file.readlines()

    # Look for the line of interest and extract the last field
    for line in lines:
        if line.startswith("[+] Done with all tasks."):
            return format_seconds(float(line.strip().split()[-1]))
    
    raise ValueError("Could not find final timestamp. Did the model finish?")

def simple_time_all():
    """
    returns a two column data-frame
    index is model code, `time` is time in seconds. 
    """
    directory = f"{data_root}/speed_test/logs/arch/"
    runs = [f for f in os.listdir(directory)]
    result = {}
    for run_name in runs:
        try:
            model_code = run_name.split("_")[0]
            result[model_code] = simple_time(run_name)
        except ValueError:
            print(f"Error on {run_name}")
    return result


In [14]:
times=pd.DataFrame(list(simple_time_all().items()),columns=["model","time"])
times.set_index("model",inplace=True)
times

,time
model,
c0010010,0h 28m 14.73s
c0010020,0h 1m 29.12s
c0011000,0h 33m 14.13s
c0011020,1h 24m 40.43s
c0012010,0h 37m 3.36s
c0100010,1h 27m 21.26s
c0100020,0h 0m 25.58s
c0200010,1h 23m 52.01s
c020020,0h 0m 58.44s


In [15]:
summary=summary.join(times,how="inner")

OK, realistically speaking we only really care about real data, perfomance on the tiny simulated dataset is great for debugging above steps, but now we don't care.

Let's also discard QC which we can't interpret for tensor

In [16]:
summary_real=summary[summary["dataset"]!="Fake (9)"]
summary_real=summary_real.drop(["covariance_matrix_available","covariance_matrix_finite","standard_errors_finite"],axis=1)

In [17]:
summary_real

,converged,too_small,total,dataset,sm_optimizer,lib,hardware,dna,main_equ_type,z_equ_type,main_equ,z_equ,broken_by,time
c0010010,10.0,0.0,10.0,Shendure (0),0_default,tensorzinb (1),0_CPU,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type,0h 28m 14.73s
c0010020,209.0,3.0,212.0,Shendure (0),0_default,tensorzinb (1),0_CPU,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id,0h 1m 29.12s
c0011000,1.0,0.0,1.0,Shendure (0),0_default,tensorzinb (1),1_A100,No dna (0),simple addition (0),replicate (0),umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),NaN,0h 33m 14.13s
c0011020,209.0,3.0,212.0,Shendure (0),0_default,tensorzinb (1),1_A100,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id,1h 24m 40.43s
c0012010,10.0,0.0,10.0,Shendure (0),0_default,tensorzinb (1),2_A100,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type,0h 37m 3.36s
c0100010,10,0.0,10.0,Shendure (0),1_bfgs,statsmodels (0),0_CPU,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type,1h 27m 21.26s
c0100020,196,0.0,212.0,Shendure (0),1_bfgs,statsmodels (0),0_CPU,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id,0h 0m 25.58s
c0200010,10,0.0,10.0,Shendure (0),2_cg,statsmodels (0),0_CPU,No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type,1h 23m 52.01s
c020020,195,0.0,212.0,Shendure (0),2_cg,statsmodels (0),0_CPU,No dna (0),by cre (2),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cre_id,0h 0m 58.44s


# How does statsmodels stack up w/r/t tensorzinb, when both are on cpu?

## cell type models

In [18]:
subs=["converged","too_small","total","sm_optimizer","dataset","lib","broken_by","time","hardware"]

In [19]:
summary_real[(summary_real["hardware"]=="0_CPU") & (summary_real["broken_by"] =="cell_type")][subs]

,converged,too_small,total,sm_optimizer,dataset,lib,broken_by,time,hardware
c0010010,10.0,0.0,10.0,0_default,Shendure (0),tensorzinb (1),cell_type,0h 28m 14.73s,0_CPU
c0100010,10,0.0,10.0,1_bfgs,Shendure (0),statsmodels (0),cell_type,1h 27m 21.26s,0_CPU
c0200010,10,0.0,10.0,2_cg,Shendure (0),statsmodels (0),cell_type,1h 23m 52.01s,0_CPU


Same convergence, tensor is faster.
(missing gc)

## CRE models

In [20]:
summary_real[(summary_real["hardware"]=="0_CPU") & (summary_real["broken_by"] =="cre_id")][subs]

,converged,too_small,total,sm_optimizer,dataset,lib,broken_by,time,hardware
c0010020,209.0,3.0,212.0,0_default,Shendure (0),tensorzinb (1),cre_id,0h 1m 29.12s,0_CPU
c0100020,196,0.0,212.0,1_bfgs,Shendure (0),statsmodels (0),cre_id,0h 0m 25.58s,0_CPU
c020020,195,0.0,212.0,2_cg,Shendure (0),statsmodels (0),cre_id,0h 0m 58.44s,0_CPU


tensorzinb I think wins: a few more models converge, at the expense of a bit more time.

(note that "too small" is same across all 3x, it's just that that counter wasn't implemented until later...)

# CPU vs GPU

## cell type models

In [21]:
summary_real[(summary_real["broken_by"] =="cell_type") & (summary_real["lib"]=="tensorzinb (1)")][subs]

,converged,too_small,total,sm_optimizer,dataset,lib,broken_by,time,hardware
c0010010,10.0,0.0,10.0,0_default,Shendure (0),tensorzinb (1),cell_type,0h 28m 14.73s,0_CPU
c0012010,10.0,0.0,10.0,0_default,Shendure (0),tensorzinb (1),cell_type,0h 37m 3.36s,2_A100


## cre models

In [22]:
summary_real[(summary_real["broken_by"] =="cre_id") & (summary_real["lib"]=="tensorzinb (1)")][subs]

,converged,too_small,total,sm_optimizer,dataset,lib,broken_by,time,hardware
c0010020,209.0,3.0,212.0,0_default,Shendure (0),tensorzinb (1),cre_id,0h 1m 29.12s,0_CPU
c0011020,209.0,3.0,212.0,0_default,Shendure (0),tensorzinb (1),cre_id,1h 24m 40.43s,1_A100


# tab for copypaste